# K-means single iteration
stough 202-

<img src="../dip_figs/headers/demo_kmeans_2D.jpg" alt="Preview: K-means single iteration" width="600"/>

Demo on k means clustering. We generate a little data and then do a single
step of the EM scheme for clustering.

Finding objects in images:
- Color coherence
- Spatial coherence

Pixels belonging to the same object tend to have similar colors (color coherence) and to be near one another (spatial coherence). **Clustering** groups data points that are similar, so it's a natural first step toward finding objects. [K-means](https://en.wikipedia.org/wiki/K-means_clustering) is the simplest and most widely used clustering method. Given a number of clusters $K$, it looks for $K$ cluster centers $\mathbf{c}_1, \ldots, \mathbf{c}_K$ and an assignment of every point to a cluster that minimize the total squared distance from each point to its center:

\begin{equation*}
J = \sum_{i=1}^{N} \left\| \mathbf{x}_i - \mathbf{c}_{k(i)} \right\|^2
\end{equation*}

where $k(i)$ is the cluster point $i$ is assigned to. We'll first see the algorithm on easy 2D data, then on the [colors of an image](./demo_kmeans_image.ipynb), and finally in [SLIC superpixels](./slic_demo.ipynb), which clusters on color *and* position.

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
from numpy import random

# For importing from alternative directory sources
import sys  
sys.path.insert(0, '../dip_utils')

from matrix_utils import (arr_info,
                          make_linmap)
from vis_utils import (vis_rgb_cube,
                       vis_hists,
                       vis_pair,
                       vis_surface)

from scipy.spatial.distance import cdist

K = 2
NUMPOINTS = 1000

In [ ]:
# Let's generate 2d random data
X1 = np.random.randn(NUMPOINTS//2, 2)  # Unit normal (mean 0, var 1)
X2 = np.random.randn(NUMPOINTS//2, 2)

X2[:,0] = 5 + 2*X2[:,0]
X2[:,1] = 3 + 2*X2[:,1]  # X2 data should be mean (5,3) var 4

X = np.concatenate([X1, X2], axis=0)

Two blobs of 500 points each: one centered at $(0,0)$ with variance 1, and one centered at $(5,3)$ with variance 4, so it's more spread out. We know the right answer here, which makes it easy to see how k-means does.

### Look at the generated data

In [ ]:
plt.figure()
plt.scatter(X[:,0], X[:,1])

In [ ]:
# For fun: Let's pick some random (and good) cluster colors
clusterColors = np.random.rand(K, 3) # Picking random colors for each cluster.
varsSoFar = np.var(clusterColors, axis=0) # Should be 1 x 2 of the variance of each column.

for i in range(3*K):
    tempColors = random.rand(K, 3) #Picking K random colors
    vartemp = np.var(tempColors, axis=0)
    if np.all(vartemp > varsSoFar):
        clusterColors = tempColors
        varsSoFar = vartemp

### Step 0: initialize

K-means needs a starting guess for the centers. A simple choice is $K$ distinct data points picked at random.

In [ ]:
# K-means: initialization
# pick K initial cluster centers.
# whichinit = random.randint(0, len(X), size=(K,)) # Could generate repeats.
whichinit = random.choice(len(X), size=K, replace=False)
CC = X[whichinit, :].copy() # Cluster Centers

CC_init = CC.copy()

In [ ]:
CC

In [ ]:
plt.figure(figsize=(4,4))
plt.scatter(X[:,0], X[:,1], c='gray', alpha=.5)
plt.scatter(CC[:,0], CC[:,1], s=30, c=clusterColors)

### Step 1: assign each point to its nearest center

[`cdist`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.cdist.html) computes the distance from every point (row of `X`) to every center (row of `CC`), an $N \times K$ matrix. `argmin` along each row then picks the closest center for each point. With the centers fixed, this is the assignment that makes $J$ as small as possible.

In [ ]:
# K-means: compute: for every data point determine which center is closest.
# Need some magic function that computes the distance between every row of X
# (the points) and every row of CC (the clusters).
D = cdist(X, CC, 'euclidean')
# D should be NUMPOINTS x K

whichCluster = np.argmin(D, axis=1) # NUMPOINTS x 1 of which center was closest

In [ ]:
D[:10,:]

In [ ]:
X[0,:]

In [ ]:
plt.figure(figsize=(4,4))
plt.scatter(X[:,0], X[:,1], c=clusterColors[whichCluster], alpha=.5)

With only two centers, the boundary between the clusters is a straight line, the perpendicular bisector between the two centers. Points on either side go to the nearer center, whether or not that matches the true blobs.

### Step 2: move each center to the mean of its points

Now hold the assignments fixed, and ask where each center should be to make $J$ as small as possible. The point that minimizes the total squared distance to a set of points is their mean, so each center moves to the average of the points assigned to it.

In [ ]:
# K-means: recompute the cluster centers as the mean of the data in each cluster
for c in range(K):
    CC[c,:] = np.mean(X[whichCluster == c, :], axis=0) # average of just those that were closest to c.

In [ ]:
np.mean(X[whichCluster == 0, :],axis=0)

In [ ]:
(whichCluster == 1).sum()

In [ ]:
plt.figure(figsize=(4,4))
plt.scatter(X[:,0], X[:,1], c='gray', alpha=.5)
plt.scatter(CC[:,0], CC[:,1], s=30, c=clusterColors)

### Repeat

That's one iteration: assign, then update. Now we assign again using the moved centers, then update again, and so on. Each step can only lower $J$ (or leave it unchanged), and there are only finitely many ways to assign the points, so the process must eventually stop changing: k-means always converges. Here's the second assignment:

In [ ]:
# K-means: compute: for every data point determine which center is closest.
# Need some magic function that computes the distance between every row of X
# (the points) and every row of CC (the clusters).
D = cdist(X, CC, 'euclidean')
# D should be NUMPOINTS x K

whichCluster = np.argmin(D, axis=1) # NUMPOINTS x 1 of which center was closest

In [ ]:
plt.figure(figsize=(4,4))
plt.scatter(X[:,0], X[:,1], c=clusterColors[whichCluster], alpha=.5)

A summary of the first iteration: the data, the random initial centers, and the clusters and centers after one update.

In [ ]:
f, ax = plt.subplots(1,3, figsize=(9,3), sharex=True, sharey=True)
ax[0].scatter(X[:,0], X[:,1], c='gray', s=20)
ax[0].set_title('Original Data')


ax[1].scatter(X[:,0], X[:,1], c='gray', alpha=.5, s=20)
ax[1].scatter(CC_init[:,0], CC_init[:,1], c=clusterColors, s=50)
ax[1].set_title('Initial Cluster Centers')






pointColors = clusterColors[whichCluster]
clusterEdgeColors = 1 - clusterColors # for contrast, make the cluster center edges opposite.

ax[2].scatter(X[:,0], X[:,1], c=pointColors, alpha=.5, s=20)
ax[2].scatter(CC[:,0], CC[:,1], c=clusterColors, edgecolors=clusterEdgeColors, s=50)
ax[2].set_title('Recomputed Clusters')

### Running to convergence

In practice we just loop until the assignments stop changing. This is the [EM-style](https://en.wikipedia.org/wiki/Expectation%E2%80%93maximization_algorithm) alternation in a few lines, starting again from the same initial centers:

In [ ]:
CC = CC_init.copy()
assignments = None
for it in range(100):
    newAssignments = np.argmin(cdist(X, CC), axis=1)          # step 1: assign
    if assignments is not None and np.all(newAssignments == assignments):
        break
    assignments = newAssignments
    CC = np.stack([X[assignments == c].mean(axis=0) for c in range(K)])  # step 2: update
    J = np.sum((X - CC[assignments])**2)
    print(f'iteration {it}: J = {J:9.1f}, centers = {CC.round(2).tolist()}')

plt.figure(figsize=(4,4))
plt.scatter(X[:,0], X[:,1], c=clusterColors[assignments], alpha=.5, s=20)
plt.scatter(CC[:,0], CC[:,1], c=clusterColors, edgecolors='k', s=80)
plt.title(f'Converged after {it} iterations');

$J$ decreases at every iteration, and within a handful of iterations the centers settle near the true blob centers, $(0,0)$ and $(5,3)$. They won't be exactly there, though. K-means draws the boundary halfway between centers, so the wide blob's outlying points that fall closer to the tight blob's center get assigned to it, pulling that center slightly toward the wide blob. Losing those points pushes the wide blob's center slightly *away*. K-means implicitly assumes round clusters of similar size.

Two other caveats to keep in mind:

- **Initialization matters.** K-means finds a *local* minimum of $J$, not necessarily the best one. Re-run the notebook a few times: on easy data like this it nearly always finds the same answer, but on harder data different starting centers can give different clusterings. Practical implementations run several random starts and keep the best, or use smarter seeding such as [k-means++](https://en.wikipedia.org/wiki/K-means%2B%2B).
- **You have to choose $K$.** Try `K = 3` at the top. K-means will happily split the data into three clusters whether or not three is meaningful.